# Breast Cancer Wisconsin - Classification using Machine Learning

**ML Assignment 2**

This notebook trains 5 ML classification models on the Breast Cancer Wisconsin dataset and evaluates each model using 6 metrics.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

print('All libraries imported successfully.')

## 2. Load the Dataset

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

print(f'Dataset shape: {X.shape}')
print(f'Number of features: {X.shape[1]}')
print(f'Number of instances: {X.shape[0]}')
print(f'\nClasses: 0 = {data.target_names[0]}, 1 = {data.target_names[1]}')
print(f'\nClass distribution:')
print(y.value_counts())

In [ ]:
# Preview the dataset
df = X.copy()
df['target'] = y
df.head(10)

In [ ]:
# Dataset info
print('Feature names:')
for i, col in enumerate(data.feature_names, 1):
    print(f'  {i}. {col}')

## 3. Train-Test Split and Feature Scaling

In [ ]:
# 80-20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Testing set:  {X_test.shape[0]} samples')

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('\nFeature scaling applied using StandardScaler.')

In [ ]:
# Save test data as CSV for the Streamlit app
test_df = pd.DataFrame(X_test, columns=data.feature_names)
test_df['target'] = y_test.values
test_df.to_csv('../test_data.csv', index=False)
print('Saved test_data.csv (114 samples)')

## 4. Train All 5 Classification Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=5000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes (Gaussian)': GaussianNB(),
    'Random Forest (Ensemble)': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
}

# Train each model
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    print(f'Trained: {name}')

print('\nAll 5 models trained successfully.')

## 5. Evaluate All Models — 6 Metrics Each

In [ ]:
results = {}

for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_proba = y_pred
    
    results[name] = {
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'AUC': round(roc_auc_score(y_test, y_proba), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1 Score': round(f1_score(y_test, y_pred), 4),
        'MCC': round(matthews_corrcoef(y_test, y_pred), 4),
    }

# Display the comparison table
results_df = pd.DataFrame(results).T
results_df.index.name = 'Model'
print('MODEL COMPARISON TABLE')
print('=' * 80)
results_df

## 6. Detailed Results — Classification Reports and Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(25, 4))

for idx, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=data.target_names,
                yticklabels=data.target_names,
                ax=axes[idx])
    axes[idx].set_title(name, fontsize=10)
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.suptitle('Confusion Matrices for All 5 Models', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed classification report for each model
for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    print(f'\n{"=" * 60}')
    print(f'{name}')
    print(f'{"=" * 60}')
    print(classification_report(y_test, y_pred, target_names=data.target_names))
    print(f'Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}')

## 7. Accuracy Comparison Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
bars = ax.bar(results_df.index, results_df['Accuracy'], color=colors)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Model Accuracy Comparison', fontsize=14)
ax.set_ylim(0.85, 1.0)

for bar, val in zip(bars, results_df['Accuracy']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', fontsize=10)

plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

best_model = results_df['Accuracy'].idxmax()
print(f'\nBest Model (by Accuracy): {best_model} ({results_df.loc[best_model, "Accuracy"]:.4f})')

## 8. Save Trained Models

In [ ]:
# Save the scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('Saved: scaler.pkl')

# Save each model
model_filenames = {
    'Logistic Regression': 'logistic_regression.pkl',
    'Decision Tree': 'decision_tree.pkl',
    'K-Nearest Neighbors': 'knn.pkl',
    'Naive Bayes (Gaussian)': 'naive_bayes.pkl',
    'Random Forest (Ensemble)': 'random_forest.pkl',
}

for name, model in models.items():
    with open(model_filenames[name], 'wb') as f:
        pickle.dump(model, f)
    print(f'Saved: {model_filenames[name]}')

print('\nAll models and scaler saved successfully!')

## 9. Summary

**Logistic Regression** is the best performing model on this dataset with:
- Accuracy: 98.25%
- AUC: 0.9954
- F1 Score: 0.9861
- MCC: 0.9623

The Breast Cancer Wisconsin dataset has features that are nearly linearly separable after scaling, which explains why a simpler model like Logistic Regression outperforms more complex models like Random Forest and Decision Tree.